# GIK-IceChain — Quickstart

End-to-end run of the full C1→C2→C3 pipeline using real production data stored in
the MinIO IceChunk store and CMORPH thresholds on disk. All data comes from the
actual East Africa flood-risk pipeline outputs — no synthetic arrays.
**Prerequisites**: `pip install -e '.[dev]'` from the repo root.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


def _bootstrap() -> Path:
    """Locate the repo; on Colab, clone + install it first."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "configs" / "default.yaml").exists():
            return p
    repo = Path.cwd() / "gik-icechain"
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/hashirama21/gik-icechain.git", str(repo)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{repo}[dev]"], check=True)
    return repo


REPO = _bootstrap()
DATA = REPO / "data"

# Download every prerequisite from public sources (idempotent): admin
# boundaries, CMORPH return periods, ENSO/IOD index, then threshold files.
tools = [sys.executable, str(REPO / "scripts" / "tools.py")]
subprocess.run([*tools, "download", "--component", "all"], check=True)
subprocess.run([*tools, "download-thresholds"], check=True)

from gik_icechain.shared.config import load_config

# Optional .env credentials (live MinIO store; offline mode works without)
_env = REPO / ".env"
if _env.exists():
    for _line in _env.read_text().splitlines():
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("AWS_ACCESS_KEY_ID", os.environ.get("MINIO_ACCESS_KEY", ""))
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", os.environ.get("MINIO_SECRET_KEY", ""))

cfg = load_config(REPO / "configs" / "default.yaml")
STORAGE_OPTIONS = {"endpoint_url": cfg.outputs.endpoint_url}

START = date(2025, 1, 1)
END   = date(2025, 1, 2)
print(f"Repo: {REPO}")
print(f"Config loaded — endpoint: {cfg.outputs.endpoint_url}")
print(f"Demo window: {START} → {END}")

## 1. C1 — IceChunk store

In [ ]:
from gik_icechain.conversion.icechunk_writer import IceChainStore

LIVE_STORE = bool(cfg.outputs.endpoint_url)
store = None
if LIVE_STORE:
    try:
        store = IceChainStore(
            cfg.outputs.icechunk_store_uri,
            region=cfg.outputs.icechunk_store_region,
            endpoint_url=cfg.outputs.endpoint_url,
        )
        store.create_or_open()
        snapshots = store.list_snapshots()
        print(f"Available snapshots: {len(snapshots)}")
        report = store.validate()
        print(f"Committed days : {report['committed_days']}  |  range {report['date_range']}")
    except Exception as exc:
        store = None
        print(f"Store error ({type(exc).__name__}: {str(exc)[:70]})")
if store is None:
    print("Offline demo mode -- no live IceChunk store (set outputs.endpoint_url to go live)")

## 2. Inspect a forecast day

In [ ]:
def _synthetic_day_ds():
    """Synthetic mm-scale ensemble tp for offline/CI demo (no live store)."""
    rng = np.random.default_rng(0)
    nm, ns, ny, nx = 51, 8, 30, 30
    tp = np.cumsum(rng.exponential(8.0, (nm, ns, ny, nx)), axis=1).astype("float32")
    return xr.Dataset(
        {"tp": (["member", "step", "latitude", "longitude"], tp)},
        coords={"member": np.arange(nm), "step": np.arange(0, ns * 6, 6),
                "latitude": np.linspace(23, -13, ny), "longitude": np.linspace(22, 53, nx)},
    )

if store is not None:
    session = store.readonly_session()
    day_ds = xr.open_zarr(session.store, group="2025-01-01", consolidated=False)
else:
    print("Offline: synthetic mm-scale demo data")
    day_ds = _synthetic_day_ds()

print(f"tp shape : {day_ds['tp'].shape}")
print(f"member   : {day_ds.sizes.get('member')}")
print(f"step     : {day_ds.sizes.get('step')}")

## 3. C2 — Exceedance probabilities

In [ ]:
from gik_icechain.exceedance.accumulations import compute_rolling_accumulations
from gik_icechain.exceedance.thresholds import (
    AdaptiveGEVThresholds, ClimateMode,
    classify_enso, classify_iod, get_season,
)
from gik_icechain.exceedance.exceedance import (
    compute_exceedance_probabilities, compute_ensemble_confidence,
)

thresholds = AdaptiveGEVThresholds.load(DATA / "cmorph_thresholds")

acc = compute_rolling_accumulations(day_ds[["tp"]], windows_h=[24, 72, 168])

enso_iod = (pd.read_csv(DATA / "enso_iod_index.csv", parse_dates=["date"])
            .set_index("date").sort_index())
row    = enso_iod.loc[enso_iod.index.asof(pd.Timestamp(START))]  # monthly index
enso   = classify_enso(float(row["nino34_anom"]))
iod    = classify_iod(float(row["dmi"]))
season = get_season(START.month)
mode   = ClimateMode(season, enso, iod)
print(f"Climate mode: {mode.key}")

thr = thresholds.get(window_h=24, return_period=5, mode=mode)
p   = compute_exceedance_probabilities(acc, xr.Dataset({"rp_5y": thr}), 24, 5, "member")
print(f"Exceedance 24h/5yr — min={float(p.min()):.3f}  max={float(p.max()):.3f}  mean={float(p.mean()):.3f}")

## 4. C3 — CRMA risk inference

In [ ]:
from gik_icechain.risk.crma_model import CRMAModel, CRMAEvidence

model = CRMAModel(crma_cfg=cfg.component3.crma_model)
model.build()

p_mean  = float(p.mean())
acc_72  = compute_rolling_accumulations(day_ds[["tp"]], windows_h=[72])
thr_72  = thresholds.get(window_h=72, return_period=5, mode=mode)
p_72    = compute_exceedance_probabilities(acc_72, xr.Dataset({"rp_5y": thr_72}), 72, 5, "member")
acc_168 = compute_rolling_accumulations(day_ds[["tp"]], windows_h=[168])
thr_168 = thresholds.get(window_h=168, return_period=5, mode=mode)
p_168   = compute_exceedance_probabilities(acc_168, xr.Dataset({"rp_5y": thr_168}), 168, 5, "member")

evidence = CRMAEvidence(
    exceedance_prob_24h=p_mean,
    exceedance_prob_72h=float(p_72.mean()),
    exceedance_prob_7d=float(p_168.mean()),
    gpm_obs_24h=0.0,
    api_mm=20.0,
    spatial_coverage_fraction=float((p > 0.15).mean()),
    consecutive_signal_days=1,
    sat_consecutive_days=0,
)
result = model.infer(evidence)
print(f"Risk label : {result['risk_label']}")
print(f"Green={result['p_green']:.3f}  Yellow={result['p_yellow']:.3f}  "
      f"Orange={result['p_orange']:.3f}  Red={result['p_red']:.3f}")

## 5. Exceedance store (C2 output)

In [ ]:
try:
    exc_ds = xr.open_zarr(
        cfg.outputs.exceedance_store_uri,
        consolidated=False,
        storage_options=STORAGE_OPTIONS,
    )
    print("Exceedance store available:")
    print(exc_ds)
except Exception as exc:
    print(f"C2 store not yet available — run gik-icechain exceedance first.")
    print(f"({type(exc).__name__}: {exc})")

Pipeline complete — see notebooks 01–03 for per-component deep dives.